# Aprendizaje por Refuerzo: aprender haciendo
**Machine Learning · Universidad Libre** — Prof. Gustavo Santos

En aprendizaje supervisado alguien nos da la respuesta correcta. En **aprendizaje por refuerzo (RL)** nadie la da:
un **agente** actúa en un **entorno**, recibe **recompensas** y aprende, por ensayo y error, qué hacer en cada situación.

**Ruta del cuaderno**
1. El dilema exploración vs. explotación: el bandido multibrazo
2. Un entorno con estados: el robot de bodega (GridWorld)
3. Planear con el modelo: iteración de valores (ecuación de Bellman)
4. Aprender sin modelo: Q-learning
5. Q-learning vs. SARSA: el acantilado
6. Taller: experimente con los hiperparámetros

> Solo usa `numpy` y `matplotlib` (vienen en Colab). Ejecute en orden con **Shift + Enter**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROJO, NARANJA, NEGRO, GRIS = "#C52646", "#DF9C30", "#212121", "#A8A8A8"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#BDBDBD", "axes.titleweight": "bold", "font.size": 11,
})
SEMILLA = 2026

## 1. Exploración vs. explotación: el bandido multibrazo
Un casino tiene **5 máquinas**. Cada una paga en promedio una cantidad distinta, pero **el agente no lo sabe**.
Tiene 1.000 jugadas. ¿Juega siempre en la que mejor le ha ido (**explotar**) o prueba otras (**explorar**)?

Estrategia **ε-greedy**: con probabilidad $\varepsilon$ elige una máquina al azar; si no, la de mayor valor estimado.

El valor estimado de cada máquina se actualiza con un promedio incremental:
$$Q_{n+1}(a) = Q_n(a) + \frac{1}{n}\big(R_n - Q_n(a)\big)$$
*nuevo estimado = estimado viejo + paso × (lo que pasó − lo que esperaba)*. Esta forma de "corregir por el error" aparece en todo RL.

In [ ]:
def bandido(epsilon, pagos_reales, jugadas=1000, rng=None):
    """Juega 'jugadas' veces con estrategia ε-greedy. Devuelve recompensas y si eligió la mejor máquina."""
    k = len(pagos_reales)
    Q = np.zeros(k)        # valor estimado de cada máquina
    N = np.zeros(k)        # veces que se jugó cada máquina
    mejor = np.argmax(pagos_reales)
    recompensas, acierto = np.zeros(jugadas), np.zeros(jugadas)
    for t in range(jugadas):
        if rng.random() < epsilon:
            a = rng.integers(k)                 # explorar
        else:
            a = rng.choice(np.flatnonzero(Q == Q.max()))  # explotar (empates al azar)
        r = rng.normal(pagos_reales[a], 1.0)    # la máquina paga con ruido
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]               # promedio incremental
        recompensas[t], acierto[t] = r, (a == mejor)
    return recompensas, acierto, Q, N

# Repetimos el experimento 300 veces (300 casinos distintos) para promediar la suerte
rng = np.random.default_rng(SEMILLA)
epsilons = [0.0, 0.01, 0.1, 0.3]
resultados = {}
for eps in epsilons:
    R, A = [], []
    for _ in range(300):
        pagos = rng.normal(0, 1, 5)
        r, a, _, _ = bandido(eps, pagos, rng=rng)
        R.append(r); A.append(a)
    resultados[eps] = (np.mean(R, axis=0), np.mean(A, axis=0) * 100)

colores = [GRIS, NEGRO, ROJO, NARANJA]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
for eps, c in zip(epsilons, colores):
    r, a = resultados[eps]
    suav = np.convolve(r, np.ones(20) / 20, mode="valid")
    axs[0].plot(suav, color=c, lw=2, label=f"ε = {eps}")
    axs[1].plot(a, color=c, lw=2, label=f"ε = {eps}")
axs[0].set(title="Recompensa promedio por jugada", xlabel="Jugada", ylabel="Recompensa")
axs[1].set(title="% de veces que eligió la mejor máquina", xlabel="Jugada", ylabel="%")
axs[1].legend(frameon=False)
plt.tight_layout(); plt.show()

for eps in epsilons:
    r, a = resultados[eps]
    print(f"ε = {eps:<4}  recompensa media = {r.mean():.3f}   mejor máquina en las últimas 100 = {a[-100:].mean():.1f} %")

**Lectura:** con $\varepsilon = 0$ el agente se "casa" con la primera máquina que le pagó bien: solo acierta la mejor el 58 % de las veces.
Con $\varepsilon = 0.1$ la descubre más rápido (86 % al final); con $\varepsilon = 0.01$ la descubre más lento pero desperdicia menos jugadas.
Demasiada exploración ($\varepsilon = 0.3$) gana menos: sigue probando máquinas malas aunque ya sabe cuál es la mejor.

**Pregunta para el grupo:** ¿en qué decisiones de su vida o de una empresa aparece este dilema?

**Respuesta.** En Odoo me pasa cuando un flujo de inventario o facturación ya “funciona” (explotar) y alguien propone un módulo nuevo o un refactor (explorar). Si nunca pruebo, me quedo con el primer enfoque que dio resultado, aunque no sea el mejor. Si exploro todo el tiempo —cambiar de librería cada sprint— pago el costo de máquinas malas: regresiones y tiempo. En la práctica uso un ε bajo: dejo un porcentaje chico de capacidad para probar, y el resto lo dejo en lo que ya medí que funciona.


## 2. Un entorno con estados: el robot de bodega
Ahora las acciones tienen **consecuencias futuras**. Un robot se mueve en una bodega de 5×6 celdas:

| Elemento | Recompensa |
|---|---|
| Cada paso | −1 (gasta batería) |
| Llegar al **despacho** (D) | +20, termina el episodio |
| Caer en la **zona de carga pesada** (X) | −20, termina el episodio |
| Paredes (■) | no se puede pasar |

Acciones: ↑ ↓ ← →. Con probabilidad 10 % el robot **resbala** y se mueve en una dirección al azar (entorno estocástico).

Esto es un **Proceso de Decisión de Markov (MDP)**: $(S, A, P, R, \gamma)$ = estados, acciones, probabilidades de transición, recompensas y factor de descuento.

In [ ]:
class Bodega:
    """GridWorld sencillo. Estados = celdas (fila, col) numeradas 0..filas*cols-1."""
    ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # arriba, abajo, izquierda, derecha
    FLECHAS = ["↑", "↓", "←", "→"]

    def __init__(self, resbalon=0.1):
        self.mapa = [
            "S...#.",
            ".##...",
            "...#X.",
            ".#....",
            "...#.D",
        ]
        self.filas, self.cols = len(self.mapa), len(self.mapa[0])
        self.n_estados, self.n_acciones = self.filas * self.cols, 4
        self.resbalon = resbalon
        self.inicio = self._buscar("S")

    def _buscar(self, ch):
        for i, fila in enumerate(self.mapa):
            if ch in fila:
                return i * self.cols + fila.index(ch)

    def celda(self, s):
        return self.mapa[s // self.cols][s % self.cols]

    def terminal(self, s):
        return self.celda(s) in "DX"

    def _mover(self, s, a):
        f, c = divmod(s, self.cols)
        df, dc = self.ACCIONES[a]
        nf, nc = f + df, c + dc
        if 0 <= nf < self.filas and 0 <= nc < self.cols and self.mapa[nf][nc] != "#":
            return nf * self.cols + nc
        return s                                     # choca y se queda

    def recompensa(self, s2):
        return {"D": 20, "X": -20}.get(self.celda(s2), -1)

    def transiciones(self, s, a):
        """Lista de (probabilidad, siguiente_estado, recompensa): el MODELO del entorno."""
        out = []
        for b in range(4):
            p = (1 - self.resbalon) * (a == b) + self.resbalon / 4
            s2 = self._mover(s, b)
            out.append((p, s2, self.recompensa(s2)))
        return out

    def paso(self, s, a, rng):
        """Simula un paso: lo único que ve un agente SIN modelo."""
        if rng.random() < self.resbalon:
            a = rng.integers(4)
        s2 = self._mover(s, a)
        return s2, self.recompensa(s2), self.terminal(s2)


def dibujar(env, V=None, politica=None, ruta=None, titulo="", ax=None):
    """Dibuja la bodega con valores (colores), política (flechas) y una ruta opcional."""
    ax = ax or plt.gca()
    cmap = LinearSegmentedColormap.from_list("u", [ROJO, "#FFFFFF", NARANJA])
    M = np.zeros((env.filas, env.cols)) if V is None else V.reshape(env.filas, env.cols).astype(float).copy()
    for s in range(env.n_estados):
        if env.celda(s) == "#":
            M[s // env.cols, s % env.cols] = np.nan
    lim = np.nanmax(np.abs(M)) if V is not None else 1
    ax.imshow(M, cmap=cmap, vmin=-lim, vmax=lim)
    for s in range(env.n_estados):
        f, c = divmod(s, env.cols); ch = env.celda(s)
        if ch == "#":
            ax.add_patch(plt.Rectangle((c - .5, f - .5), 1, 1, color=NEGRO))
        elif ch in "DX":
            ax.text(c, f, "D" if ch == "D" else "X", ha="center", va="center", fontsize=18, fontweight="bold",
                    color=NEGRO if ch == "D" else ROJO)
        else:
            txt = env.FLECHAS[politica[s]] if politica is not None else ""
            ax.text(c, f - .08, txt, ha="center", va="center", fontsize=17, color=NEGRO)
            if V is not None:
                ax.text(c, f + .3, f"{V[s]:.1f}", ha="center", va="center", fontsize=8, color="#595959")
    if ruta is not None:
        pts = np.array([divmod(s, env.cols) for s in ruta])
        ax.plot(pts[:, 1], pts[:, 0], color="#3B6E9A", lw=3, alpha=.8)
        ax.scatter(pts[0, 1], pts[0, 0], s=120, color="#3B6E9A", zorder=3)
    ax.set_xticks(np.arange(-.5, env.cols), []); ax.set_yticks(np.arange(-.5, env.filas), [])
    ax.grid(color="white" if V is not None else "#CCCCCC", lw=2); ax.tick_params(length=0)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title(titulo)


env = Bodega()
plt.figure(figsize=(6, 5)); dibujar(env, titulo="La bodega (S = salida del robot)")
f, c = divmod(env.inicio, env.cols); plt.text(c, f, "S", ha="center", va="center", fontsize=18, fontweight="bold", color="#3B6E9A")
plt.show()
print("Estados:", env.n_estados, "| Acciones:", env.n_acciones)
print("Modelo desde la salida, acción → :", env.transiciones(env.inicio, 3))

## 3. Planear con el modelo: iteración de valores
El agente busca maximizar el **retorno**: la suma de recompensas futuras, descontadas con $\gamma \in [0,1)$:
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots$$

El **valor** de un estado es el retorno esperado si desde allí actuamos lo mejor posible. Cumple la **ecuación de optimalidad de Bellman**:
$$V^*(s) = \max_a \sum_{s'} P(s' \mid s,a)\,\big[R(s,a,s') + \gamma V^*(s')\big]$$

Si **conocemos el modelo** $P$ y $R$, podemos aplicar esa ecuación una y otra vez hasta que los valores no cambien: **iteración de valores**.
Luego la política óptima es elegir en cada estado la acción que logra el máximo.

In [ ]:
def iteracion_valores(env, gamma=0.95, tol=1e-6):
    V = np.zeros(env.n_estados)
    historia = []
    while True:
        delta, V_nuevo = 0, V.copy()
        for s in range(env.n_estados):
            if env.terminal(s) or env.celda(s) == "#":
                continue
            q = [sum(p * (r + gamma * V[s2] * (not env.terminal(s2))) for p, s2, r in env.transiciones(s, a))
                 for a in range(env.n_acciones)]
            V_nuevo[s] = max(q)
            delta = max(delta, abs(V_nuevo[s] - V[s]))
        V = V_nuevo
        historia.append(delta)
        if delta < tol:
            break
    politica = np.zeros(env.n_estados, dtype=int)
    for s in range(env.n_estados):
        q = [sum(p * (r + gamma * V[s2] * (not env.terminal(s2))) for p, s2, r in env.transiciones(s, a))
             for a in range(env.n_acciones)]
        politica[s] = int(np.argmax(q))
    return V, politica, historia

V_opt, pi_opt, hist = iteracion_valores(env)
print(f"Convergió en {len(hist)} iteraciones")

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.2, 1]})
dibujar(env, V_opt, pi_opt, titulo="Valor óptimo V*(s) y política óptima", ax=axs[0])
axs[1].semilogy(hist, color=ROJO, lw=2)
axs[1].set(title="Cambio máximo por iteración (escala log)", xlabel="Iteración", ylabel="max |ΔV|")
plt.tight_layout(); plt.show()

**Lectura:** los valores crecen a medida que nos acercamos al despacho, y las flechas rodean la zona X.
Pero hubo trampa: usamos el **modelo** (`env.transiciones`). En la vida real casi nunca lo conocemos.

## 4. Aprender sin modelo: Q-learning
Ahora el agente **solo puede jugar** (`env.paso`) y observar $(s, a, r, s')$. Aprende la tabla $Q(s,a)$ = valor de hacer $a$ en $s$:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\Big[\underbrace{r + \gamma \max_{a'} Q(s',a')}_{\text{objetivo}} - Q(s,a)\Big]$$

- $\alpha$ = tasa de aprendizaje (cuánto creerle a la nueva experiencia)
- $\gamma$ = factor de descuento (cuánto importa el futuro)
- $\varepsilon$ = exploración (igual que en el bandido), que vamos reduciendo con los episodios

Es la misma idea del bandido —*corregir por el error*— pero el objetivo incluye el valor del estado siguiente.

In [ ]:
def q_learning(env, episodios=3000, alpha=0.1, gamma=0.95, eps_ini=1.0, eps_fin=0.05, max_pasos=200, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((env.n_estados, env.n_acciones))
    retornos = []
    for ep in range(episodios):
        eps = max(eps_fin, eps_ini * (1 - ep / (0.7 * episodios)))   # ε decrece linealmente
        s, total = env.inicio, 0
        for _ in range(max_pasos):
            a = rng.integers(4) if rng.random() < eps else rng.choice(np.flatnonzero(Q[s] == Q[s].max()))
            s2, r, fin = env.paso(s, a, rng)
            objetivo = r + (0 if fin else gamma * Q[s2].max())
            Q[s, a] += alpha * (objetivo - Q[s, a])          # ← la actualización de Q-learning
            s, total = s2, total + r
            if fin:
                break
        retornos.append(total)
    return Q, np.array(retornos)

Q, retornos = q_learning(env)
pi_q = Q.argmax(axis=1)
V_q = Q.max(axis=1)

def ruta_greedy(env, pi, max_pasos=40):
    s, ruta = env.inicio, [env.inicio]
    for _ in range(max_pasos):
        s = env._mover(s, pi[s]); ruta.append(s)
        if env.terminal(s):
            break
    return ruta

fig, axs = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.2]})
media = np.convolve(retornos, np.ones(100) / 100, mode="valid")
axs[0].plot(retornos, color=GRIS, alpha=.3, lw=.8)
axs[0].plot(np.arange(99, len(retornos)), media, color=ROJO, lw=2.5, label="media móvil 100 episodios")
axs[0].set(title="Curva de aprendizaje", xlabel="Episodio", ylabel="Retorno del episodio"); axs[0].legend(frameon=False)
dibujar(env, V_q, pi_q, ruta=ruta_greedy(env, pi_q), titulo="Lo que aprendió Q-learning (línea = ruta del robot)", ax=axs[1])
plt.tight_layout(); plt.show()

coinciden = [s for s in range(env.n_estados) if env.celda(s) in "S." and pi_q[s] == pi_opt[s]]
libres = [s for s in range(env.n_estados) if env.celda(s) in "S."]
print(f"La política aprendida coincide con la óptima en {len(coinciden)} de {len(libres)} celdas libres")
print(f"Retorno medio últimos 300 episodios: {retornos[-300:].mean():.2f}")

**Lectura:** al inicio el robot vaga y cae en X (retornos muy negativos). Con la experiencia, la curva sube y se estabiliza.
Sin conocer el modelo, Q-learning llega a una política casi idéntica a la de iteración de valores.

Veamos la tabla Q de la celda de salida: una fila por estado, una columna por acción.

In [ ]:
import pandas as pd
tabla = pd.DataFrame(Q[[env.inicio, 1, 6, 12]], columns=["↑", "↓", "←", "→"],
                     index=["salida (0,0)", "(0,1)", "(1,0)", "(2,0)"]).round(2)
tabla

## 5. Q-learning vs. SARSA: el acantilado
Un caminante va de **S** a **M** bordeando un acantilado. Caer cuesta −100 y lo devuelve al inicio; cada paso cuesta −1.

**SARSA** usa la acción que realmente va a tomar (incluida la exploración):
$$Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma\,Q(s',a') - Q(s,a)\big]$$
**Q-learning** usa la mejor acción posible, $\max_{a'} Q(s',a')$, aunque luego explore.

- Q-learning es **off-policy**: aprende la política óptima aunque se comporte explorando.
- SARSA es **on-policy**: aprende el valor de lo que realmente hace, riesgos de explorar incluidos.

In [ ]:
class Acantilado:
    filas, cols, n_acciones = 4, 12, 4
    ACCIONES = Bodega.ACCIONES
    FLECHAS = Bodega.FLECHAS
    def __init__(self):
        self.n_estados = self.filas * self.cols
        self.inicio, self.meta = 3 * self.cols, 3 * self.cols + self.cols - 1
    def es_acantilado(self, s):
        return s // self.cols == 3 and 0 < s % self.cols < self.cols - 1
    def paso(self, s, a, rng=None):
        f, c = divmod(s, self.cols); df, dc = self.ACCIONES[a]
        nf, nc = min(max(f + df, 0), self.filas - 1), min(max(c + dc, 0), self.cols - 1)
        s2 = nf * self.cols + nc
        if self.es_acantilado(s2):
            return self.inicio, -100, False
        return s2, -1, s2 == self.meta


def entrenar(env, metodo, episodios=500, alpha=0.5, gamma=1.0, eps=0.1, semilla=0):
    rng = np.random.default_rng(semilla)
    Q = np.zeros((env.n_estados, 4)); retornos = []
    elegir = lambda s: rng.integers(4) if rng.random() < eps else rng.choice(np.flatnonzero(Q[s] == Q[s].max()))
    for _ in range(episodios):
        s, total = env.inicio, 0
        a = elegir(s)
        for _ in range(500):
            s2, r, fin = env.paso(s, a)
            a2 = elegir(s2)
            if metodo == "sarsa":
                objetivo = r + (0 if fin else gamma * Q[s2, a2])
            else:
                objetivo = r + (0 if fin else gamma * Q[s2].max())
            Q[s, a] += alpha * (objetivo - Q[s, a])
            s, a, total = s2, a2, total + r
            if fin:
                break
        retornos.append(total)
    return Q, np.array(retornos)

cliff = Acantilado()
curvas, Qs = {}, {}
for m in ["qlearning", "sarsa"]:
    runs = [entrenar(cliff, m, semilla=k) for k in range(20)]
    curvas[m] = np.mean([r for _, r in runs], axis=0)
    Qs[m] = runs[1][0]

def ruta_cliff(env, Q):
    s, ruta = env.inicio, [env.inicio]
    for _ in range(60):
        s, _, fin = env.paso(s, int(Q[s].argmax())); ruta.append(s)
        if fin:
            break
    return ruta

fig = plt.figure(figsize=(12, 7.2))
ax0 = fig.add_subplot(2, 1, 1)
for m, c, lab in [("qlearning", ROJO, "Q-learning"), ("sarsa", NARANJA, "SARSA")]:
    ax0.plot(np.convolve(curvas[m], np.ones(10) / 10, mode="valid"), color=c, lw=2.5, label=lab)
ax0.set_ylim(-110, 0); ax0.legend(frameon=False)
ax0.set(title="Retorno por episodio mientras exploran (ε = 0.1)", xlabel="Episodio", ylabel="Retorno")
ax1 = fig.add_subplot(2, 1, 2)
M = np.zeros((cliff.filas, cliff.cols))
M[3, 1:-1] = 1
ax1.imshow(M, cmap=LinearSegmentedColormap.from_list("c", ["#FFFFFF", "#F4C6CF"]))
for m, c, off in [("qlearning", ROJO, .12), ("sarsa", NARANJA, -.12)]:
    pts = np.array([divmod(s, cliff.cols) for s in ruta_cliff(cliff, Qs[m])])
    ax1.plot(pts[:, 1], pts[:, 0] + off, color=c, lw=3.5, label="Q-learning" if m == "qlearning" else "SARSA")
ax1.text(0, 3, "S", ha="center", va="center", fontsize=16, fontweight="bold")
ax1.text(cliff.cols - 1, 3, "M", ha="center", va="center", fontsize=16, fontweight="bold")
ax1.text(5.5, 3, "ACANTILADO  (−100)", ha="center", va="center", color=ROJO, fontweight="bold")
ax1.set_xticks(np.arange(-.5, cliff.cols), []); ax1.set_yticks(np.arange(-.5, cliff.filas), [])
ax1.grid(color="#DDDDDD"); ax1.tick_params(length=0); ax1.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1))
ax1.set_title("Ruta aprendida por cada método")
plt.tight_layout(); plt.show()

print(f"Retorno medio últimos 100 episodios  Q-learning: {curvas['qlearning'][-100:].mean():.1f}   SARSA: {curvas['sarsa'][-100:].mean():.1f}")

**Lectura:** Q-learning aprende la ruta **óptima**, pegada al borde, pero mientras explora a veces cae: su retorno durante el entrenamiento es peor.
SARSA aprende una ruta **más segura**, lejos del borde, porque sabe que a veces dará un paso al azar.

**Pregunta:** si el agente fuera un carro autónomo o un robot de trading entrenando con dinero real, ¿cuál preferiría?

**Respuesta.** Si yo entrenara un agente que mueve existencias reales o dispara asientos en producción, preferiría **SARSA**. Q-learning aprende la ruta óptima pegada al riesgo (el “acantilado”: validar mal un picking, pisar una zona X). Mientras el agente sigue explorando, esa política óptima se cae. SARSA aprende lo que *de verdad hace*, exploración incluida, y se aleja del borde. Con dinero o inventario real quiero la política segura, no la más corta en el simulador.


## 6. Taller en clase
Use las funciones de arriba y responda con evidencia (gráficas o números):

1. **Bandido:** ¿qué pasa con $\varepsilon = 0.5$? ¿Y si la exploración decrece con el tiempo, $\varepsilon_t = 1/\sqrt{t}$?
2. **Descuento:** entrene Q-learning en la bodega con $\gamma = 0.5$ y con $\gamma = 0.99$. ¿Cambia la ruta? ¿Por qué?
3. **Resbalón:** cree `Bodega(resbalon=0.4)` y calcule la política óptima. ¿Se aleja más de la zona X?
4. **Recompensas:** cambie el costo por paso de −1 a 0. ¿El robot sigue buscando el camino corto? (Pista: edite `recompensa`.)
5. **Tasa de aprendizaje:** compare $\alpha = 0.01$, $0.1$ y $0.9$ en la curva de aprendizaje.

La celda siguiente trae un punto de partida para la pregunta 2.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
for ax, g in zip(axs, [0.5, 0.99]):
    Qg, _ = q_learning(Bodega(), gamma=g, episodios=5000)
    dibujar(env, Qg.max(axis=1), Qg.argmax(axis=1), ruta=ruta_greedy(env, Qg.argmax(axis=1)), titulo=f"Q-learning con γ = {g}", ax=ax)
plt.tight_layout(); plt.show()

## Respuestas del taller

Helpers locales: no modifican `bandido` ni `Bodega` de las secciones anteriores.


In [ ]:

def bandido_eps(epsilon_fn, pagos_reales, jugadas=1000, rng=None):
    """Igual que bandido, pero ε = epsilon_fn(t) con t = 1..jugadas."""
    k = len(pagos_reales)
    Q = np.zeros(k)
    N = np.zeros(k)
    mejor = np.argmax(pagos_reales)
    recompensas, acierto = np.zeros(jugadas), np.zeros(jugadas)
    for t0 in range(jugadas):
        t = t0 + 1
        if rng.random() < epsilon_fn(t):
            a = rng.integers(k)
        else:
            a = rng.choice(np.flatnonzero(Q == Q.max()))
        r = rng.normal(pagos_reales[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        recompensas[t0], acierto[t0] = r, (a == mejor)
    return recompensas, acierto, Q, N


class BodegaCosto(Bodega):
    """Bodega con costo de paso configurable. No toca Bodega.recompensa."""

    def __init__(self, costo_paso=0, resbalon=0.1):
        super().__init__(resbalon=resbalon)
        self.costo_paso = costo_paso

    def recompensa(self, s2):
        return {"D": 20, "X": -20}.get(self.celda(s2), self.costo_paso)


def manhattan_estado(env, s, t):
    f1, c1 = divmod(s, env.cols)
    f2, c2 = divmod(t, env.cols)
    return abs(f1 - f2) + abs(c1 - c2)


def media_dist_sucesor(env, politica):
    x = env._buscar("X")
    dists = [
        manhattan_estado(env, env._mover(s, int(politica[s])), x)
        for s in range(env.n_estados)
        if env.celda(s) in "S."
    ]
    return float(np.mean(dists))


def n_celdas_mas_lejos(env, pi_lo, pi_hi):
    x = env._buscar("X")
    n = 0
    for s in range(env.n_estados):
        if env.celda(s) not in "S.":
            continue
        d_lo = manhattan_estado(env, env._mover(s, int(pi_lo[s])), x)
        d_hi = manhattan_estado(env, env._mover(s, int(pi_hi[s])), x)
        if d_hi > d_lo:
            n += 1
    return n


def ruta_llego_a_terminal(env, ruta):
    return env.terminal(ruta[-1])


### 1. Bandido


In [ ]:

rng = np.random.default_rng(SEMILLA)
estrategias = {
    "ε = 0.5": lambda t: 0.5,
    "ε = 1/√t": lambda t: 1.0 / np.sqrt(t),
}
resultados_p1 = {}
for nombre, fn in estrategias.items():
    R, A = [], []
    for _ in range(300):
        pagos = rng.normal(0, 1, 5)
        r, a, _, _ = bandido_eps(fn, pagos, rng=rng)
        R.append(r)
        A.append(a)
    resultados_p1[nombre] = (np.mean(R, axis=0), np.mean(A, axis=0) * 100)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.3))
for (nombre, c) in zip(estrategias, [ROJO, NARANJA]):
    r, a = resultados_p1[nombre]
    suav = np.convolve(r, np.ones(20) / 20, mode="valid")
    axs[0].plot(suav, color=c, lw=2, label=nombre)
    axs[1].plot(a, color=c, lw=2, label=nombre)
axs[0].set(title="Recompensa promedio por jugada", xlabel="Jugada", ylabel="Recompensa")
axs[1].set(title="% de veces que eligió la mejor máquina", xlabel="Jugada", ylabel="%")
axs[1].legend(frameon=False)
plt.tight_layout()
plt.show()

for nombre in estrategias:
    r, a = resultados_p1[nombre]
    print(f"{nombre:<12}  recompensa media = {r.mean():.3f}   mejor máquina en las últimas 100 = {a[-100:].mean():.1f} %")


**Frase.** Con ε = 0.5 la recompensa media es 0.569 y acierta 58.3 % al final; con 1/√t la media es 1.023 y acierta 88.8 % porque deja de explorar cuando ya vio la mejor máquina.
